# XGBoost Default Prediction

This notebook trains an XGBoost model to predict `default payment next month`. XGBoost is a strong baseline for tabular data and is useful to compare against the MLP and later Flower federated-learning experiments.

## Required Package

If XGBoost is missing, install it in the `fairfed` environment:

```bash
conda activate fairfed
conda install -c conda-forge xgboost
```

Alternative:

```bash
pip install xgboost
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)

In [ ]:
DATA_PATHS = [
    Path("../data/default_of_credit_card_clients.xls"),
    Path("data/default_of_credit_card_clients.xls"),
]
DATA_PATH = next(path for path in DATA_PATHS if path.exists())

df = pd.read_excel(DATA_PATH, header=1)
target_col = "default payment next month"

print(df.shape)
df.head()

## Feature Engineering

For tabular models, engineered financial behavior features often help more than making the model larger. These features summarize repayment delay, bill amounts, payment amounts, utilization, and recent changes.

In [ ]:
def add_features(data):
    data = data.copy()

    pay_status_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
    bill_cols = ["BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"]
    pay_amt_cols = ["PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]

    data["MAX_PAY_DELAY"] = data[pay_status_cols].max(axis=1)
    data["AVG_PAY_DELAY"] = data[pay_status_cols].mean(axis=1)
    data["MONTHS_WITH_DELAY"] = (data[pay_status_cols] > 0).sum(axis=1)

    data["TOTAL_BILL_AMT"] = data[bill_cols].sum(axis=1)
    data["AVG_BILL_AMT"] = data[bill_cols].mean(axis=1)
    data["MAX_BILL_AMT"] = data[bill_cols].max(axis=1)

    data["TOTAL_PAY_AMT"] = data[pay_amt_cols].sum(axis=1)
    data["AVG_PAY_AMT"] = data[pay_amt_cols].mean(axis=1)
    data["MAX_PAY_AMT"] = data[pay_amt_cols].max(axis=1)

    data["PAY_TO_BILL_RATIO"] = data["TOTAL_PAY_AMT"] / (data["TOTAL_BILL_AMT"].abs() + 1)
    data["LIMIT_UTILIZATION_1"] = data["BILL_AMT1"] / (data["LIMIT_BAL"] + 1)
    data["LIMIT_UTILIZATION_AVG"] = data["AVG_BILL_AMT"] / (data["LIMIT_BAL"] + 1)

    data["BILL_CHANGE_1_2"] = data["BILL_AMT1"] - data["BILL_AMT2"]
    data["BILL_CHANGE_AVG"] = data[bill_cols].diff(axis=1).iloc[:, 1:].mean(axis=1)

    return data

df_model = add_features(df)
df_model.head()

In [ ]:
X = df_model.drop(columns=["ID", target_col])
y = df_model[target_col].astype(int)

categorical_cols = ["SEX", "EDUCATION", "MARRIAGE", "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("categorical", one_hot, categorical_cols),
        ("numeric", "passthrough", numeric_cols),
    ]
)

X_train = preprocess.fit_transform(X_train_raw)
X_val = preprocess.transform(X_val_raw)
X_test = preprocess.transform(X_test_raw)

scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Train default rate:", round(y_train.mean(), 4))
print("scale_pos_weight:", round(scale_pos_weight, 4))

## Train XGBoost

This configuration is a sensible first XGBoost baseline: enough trees to learn nonlinear interactions, early stopping to avoid overfitting, and class weighting for the imbalanced default class.

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=800,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=5,
    reg_lambda=2.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=-1,
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

val_prob = xgb_model.predict_proba(X_val)[:, 1]
print("Validation ROC-AUC:", round(roc_auc_score(y_val, val_prob), 4))
print("Validation PR-AUC:", round(average_precision_score(y_val, val_prob), 4))

## Tune Decision Threshold

The default threshold `0.5` is not always best for imbalanced classification. This cell chooses the validation threshold that maximizes F1 for the `default = 1` class.

In [ ]:
threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 181):
    val_pred = (val_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "f1": f1_score(y_val, val_pred, zero_division=0),
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_df.sort_values(["f1", "recall"], ascending=False).iloc[0]
best_threshold = float(best_threshold_row["threshold"])

best_threshold_row

## Test Evaluation

In [ ]:
test_prob = xgb_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

print("Best threshold:", round(best_threshold, 3))
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))
print("Test PR-AUC:", round(average_precision_score(y_test, test_prob), 4))
print("Test accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Test F1:", round(f1_score(y_test, test_pred), 4))
print("Test precision:", round(precision_score(y_test, test_pred), 4))
print("Test recall:", round(recall_score(y_test, test_pred), 4))
print()
print(classification_report(y_test, test_pred, target_names=["no default", "default"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_prob)
precision, recall, _ = precision_recall_curve(y_test, test_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(fpr, tpr)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")

axes[1].plot(recall, precision)
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")

fig.tight_layout()
plt.show()

## Feature Importance

In [ ]:
feature_names = preprocess.get_feature_names_out()
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

top_importance = importance_df.head(20).sort_values("importance")

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top_importance["feature"], top_importance["importance"])
ax.set_title("Top 20 XGBoost Feature Importances")
ax.set_xlabel("Importance")
fig.tight_layout()
plt.show()

importance_df.head(20)

In [ ]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

xgb_model.save_model(MODEL_DIR / "credit_default_xgboost.json")
print("Saved model to", MODEL_DIR / "credit_default_xgboost.json")